# Train TransitEye from full TESS light-curve FITS files
Upload the prepared **TransitEye-FITS** folder to **My Drive** using Drive's folder upload. On your machine it lives under `.transiteye-workspace/colab/`. To recreate it, run `python scripts/prepare_colab_fits.py` from the project root. The local FITS entries are hard links to your archive. Treat them as read-only.

Allow about **39 GiB** in Drive plus room for results, and **40 GiB** free on the Colab VM. Choose a GPU runtime for CNN training. Run the cells in order. Raw files are copied to the VM for extraction. Per-source extraction checkpoints and model outputs are written directly to Drive so a disconnected runtime can resume extraction. A CNN training session that disconnects restarts from initialization.

The frozen catalog cohort supplies labels, event metadata and TIC-grouped train, calibration and selection assignments. Each input is a full downloaded observation. The trainer searches the flux first, matches repeated detections to catalog events, and uses the first matching observation per source to form three 256-bin channels. Unmatched targets have no supervised training label. The output classes are transit-like and eclipse-like.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
from collections import Counter

drive_root = Path('/content/drive/MyDrive')
folders = [drive_root/'colab/data', Path('/content/colab/data'), Path('/content/data'), drive_root/'TransitEye-FITS']
source = next((p for folder in folders for p in (folder, folder/'TransitEye-FITS')
               if (p/'development.json').is_file()), None)
if source is None:
    raise FileNotFoundError('Place the prepared folder containing development.json and fits/ in My Drive/colab/data or /content/data.')
print('Using data from', source)
local = Path('/content/transiteye-fits-run')
local.mkdir(exist_ok=True)
manifest_bytes = (source/'development.json').read_bytes()
manifest_hash = hashlib.sha256(manifest_bytes).hexdigest()
if manifest_hash != 'ba91ffc98c60b9a008da99291a6f5b2fa261b47f60c5c3e46dbf48f2e9434c27':
    raise ValueError('Expected the staged TransitEye development manifest. Check your Drive folder.')
products = json.loads(manifest_bytes)['products']
files = {r['relative_path']:r['expected_sha256'] for r in products}
print(len(products), 'product records;', len(files), 'distinct FITS payloads')
print(Counter(r['split'] for r in products))

## Prepare the runtime
Use Colab’s PyTorch build for the assigned accelerator. Install the scientific packages used by the FITS reader, detrending and period search. The run saves the actual environment alongside the report.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','astropy>=6,<8','wotan>=1.10,<3',
                'scikit-learn>=1.4,<2','pyarrow>=16,<27','numpy>=1.26,<3','scipy>=1.13,<2'],check=True)
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('CNN device:', device)

## Stage and verify the FITS files
This step checks the original SHA-256 hash of each light curve before training. Rerunning it on the same VM reuses verified copies.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
if not (local/'fits').exists() and shutil.disk_usage(local).free < 40*2**30:
    raise RuntimeError('This runtime needs at least 40 GiB free to stage the FITS cohort.')
for name in ('src','configs','scripts'):
    shutil.copytree(source/name,local/name,dirs_exist_ok=True)
shutil.copyfile(source/'uv.lock',local/'uv.lock')
(local/'development.json').write_bytes(manifest_bytes)

def stage_file(item):
    name, digest = item
    target = local/name
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() == digest:
        return
    payload = (source/name).read_bytes()
    if hashlib.sha256(payload).hexdigest() != digest:
        raise ValueError(f'Upload checksum mismatch: {name}')
    target.write_bytes(payload)

with ThreadPoolExecutor(max_workers=4) as pool:
    for i,_ in enumerate(pool.map(stage_file,files.items()),1):
        if i%1000 == 0:
            print(i,'/',len(files),flush=True)

## Search the full curves and train
Each extraction worker checks the FITS target, sector and checksum, filters flagged samples, normalizes and detrends each segment, then searches for repeating dips. The forest and CNN train on the same matched candidates. The selection partition chooses the CNN epoch. The calibration partition sets its temperature. Extraction can span multiple Colab sessions.

In [ ]:
import importlib.metadata as md
fingerprint = hashlib.sha256(manifest_bytes)
for p in sorted((local/'src').rglob('*.py')) + [local/'configs/labeled_candidate_sequence.v1.toml', local/'scripts/train_labeled_candidate_sequence.py']:
    fingerprint.update(p.read_bytes())
for package in ('torch','scipy','astropy','wotan','numpy','scikit-learn','numba','pyarrow'):
    fingerprint.update((package+md.version(package)).encode())
out = drive_root/'TransitEye-training-results'/fingerprint.hexdigest()[:16]
out.mkdir(parents=True,exist_ok=True)
(out/'environment.txt').write_bytes(subprocess.check_output([sys.executable,'-m','pip','freeze']))
env = dict(os.environ, PYTHONPATH=str(local/'src'), CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
subprocess.run([sys.executable,'-u','scripts/train_labeled_candidate_sequence.py',
                '--manifest','development.json','--output-dir',str(out),'--workers','2',
                '--device',device,'--batch-size','32','--maximum-epochs','100'],
               cwd=local, env=env, check=True)

## Review results
Read the matching counts before comparing model scores. Selection macro-F1 describes the matched development candidates. The historical holdout has already been evaluated.

In [ ]:
report = json.loads((out/'training_report.json').read_text())
print('Candidate extraction:', report['candidate_matching_status_counts'])
print('Forest selection macro-F1:', report['baseline']['selection']['macro_f1'])
print('CNN selection macro-F1:', report['cnn']['selection_macro_f1_at_selected_epoch'])
print('Models, checkpoints and reports:',out)